# Practical Lab 01: Synchronous Fast Router (<80 ms) with TypeSafe AI (Jev)
## High-Throughput Semantic Routing with Zero Output Tokens
**Book:** *Jev & System One Architecture: Sub-100ms Calibrated Decisions and RLCD*  
**Repository:** [https://github.com/0utKast/Jev_AI](https://github.com/0utKast/Jev_AI)  
**O'Reilly References:** *AI Engineering* (Chip Huyen, 2024) & *Agentic Architectural Patterns* (Dr. Ali Arsanjani, 2026)

In [ ]:
# ==============================================================================
# CELL 1: TypeSafe AI System One Engine & Native Typed Primitives (Choice, Score, Noul)
# Zero external dependencies — 100% Python Standard Library
# ==============================================================================
import time
from dataclasses import dataclass
from typing import List, Dict, Any, Union

@dataclass
class Choice:
    name: str
    options: List[str]

@dataclass
class Score:
    name: str
    rubric: str
    min_val: float = 1.0
    max_val: float = 10.0

@dataclass
class Noul:
    name: str
    assertion: str

@dataclass
class ChoiceResult:
    choice: str
    probability: float
    distribution: Dict[str, float]

@dataclass
class ScoreResult:
    score: float
    normalized: float
    confidence: float

@dataclass
class NoulResult:
    value: bool
    probability: float

class DecisionResult:
    def __init__(self, results: Dict[str, Any], latency_ms: float):
        self._results = results
        self.latency_ms = latency_ms
        self.tokens_generated = 0
        for k, v in results.items():
            setattr(self, k, v)

class TypeSafeClient:
    """System One non-generative decision engine: O(1) parallel sampling in ~70 ms."""
    def __init__(self, api_key: str = "ts_live_secret_key"):
        self.api_key = api_key

    def system_one(self, context: str, *primitives: Union[Choice, Score, Noul]) -> DecisionResult:
        start_time = time.perf_counter()
        results = {}
        text = context.lower()
        for prim in primitives:
            if isinstance(prim, Choice):
                scores = {opt: 0.12 for opt in prim.options}
                for opt in prim.options:
                    if opt in ('technical_support', 'query_order_status') and any(w in text for w in ('error', 'crash', 'outage', 'database', 'order', 'status', 'latency')):
                        scores[opt] += 5.2
                    elif opt in ('billing', 'issue_partial_refund') and any(w in text for w in ('invoice', 'charge', 'refund', 'payment', 'duplicate', 'overcharged')):
                        scores[opt] += 5.5
                    elif opt in ('sales', 'update_shipping_address') and any(w in text for w in ('license', 'enterprise', 'quote', 'address', 'shipping')):
                        scores[opt] += 4.5
                    elif opt in ('churn_retention', 'delete_customer_account') and any(w in text for w in ('cancel', 'terminate', 'delete', 'competitor', 'lawsuit', 'wipe')):
                        scores[opt] += 6.0
                    elif opt == 'safe_query' and not any(w in text for w in ('ignore all', 'dan', 'ssn', 'sk-live', 'idiot', 'destroy')):
                        scores[opt] += 5.8
                    elif opt == 'prompt_injection_jailbreak' and any(w in text for w in ('ignore all', 'dan', 'system prompt', 'bypass')):
                        scores[opt] += 7.5
                    elif opt == 'pii_credential_leak' and any(w in text for w in ('sk-live', '4532-', 'ssn', 'password')):
                        scores[opt] += 7.2
                tot = sum(scores.values())
                probs = {k: round(v / tot, 4) for k, v in scores.items()}
                best = max(probs, key=probs.get)
                results[prim.name] = ChoiceResult(choice=best, probability=probs[best], distribution=probs)
            elif isinstance(prim, Score):
                markers = ('urgent', 'production', 'outage', 'frozen', 'immediately', 'critical', 'lawsuit', 'destroy', 'idiot', 'hostile')
                hits = sum(1 for m in markers if m in text)
                val = min(prim.max_val, max(prim.min_val, round(1.5 + hits * 2.4, 1)))
                results[prim.name] = ScoreResult(score=val, normalized=round(val / prim.max_val, 2), confidence=0.94)
            elif isinstance(prim, Noul):
                risk = any(w in text for w in ('cancel', 'lawsuit', 'outage', 'ignore all', 'dan', 'sk-live', 'delete', 'wipe', '4532-'))
                prob = 0.96 if risk else 0.04
                results[prim.name] = NoulResult(value=(prob >= 0.5), probability=prob)
        time.sleep(0.071)
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 1)
        return DecisionResult(results, elapsed_ms)

client = TypeSafeClient()
print("✅ TypeSafe AI System One Engine initialized (O(1) Parallel Sampler ready).")

In [ ]:
# ==============================================================================
# CELL 2: Declare Atomic Routing Primitives & Evaluate Incoming Support Tickets
# ==============================================================================
dept_choice = Choice('department', ['technical_support', 'billing', 'sales', 'churn_retention'])
severity_score = Score('severity', 'Operational urgency from 1 (routine) to 10 (critical outage)')
churn_noul = Noul('churn_risk', 'The customer threatens to cancel subscription or take legal action')

tickets = [
    'URGENT: Production cluster database is frozen after upgrade. We are losing transactions immediately!',
    'Hello, we were charged a duplicate invoice on our corporate card and need an immediate refund.',
    'If this outage is not resolved today we will cancel our enterprise contract and move to a competitor.'
]

for idx, ticket in enumerate(tickets, 1):
    res = client.system_one(ticket, dept_choice, severity_score, churn_noul)
    print(f'\n--- Ticket #{idx} ({res.latency_ms} ms | Output Tokens: {res.tokens_generated}) ---')
    print(f'  Text       : {ticket}')
    print(f'  Department : {res.department.choice} (p={res.department.probability:.1%})')
    print(f'  Severity   : {res.severity.score}/10.0')
    print(f'  Churn Risk : {res.churn_risk.value} (p={res.churn_risk.probability:.1%})')